In [62]:
import os
import pandas as pd

train_root = 'data/train'  # Update path if needed
data = []

for class_folder in sorted(os.listdir(train_root)):
    class_path = os.path.join(train_root, class_folder)
    if os.path.isdir(class_path):
        # Extract numeric part from folder name (e.g., "001" from "001.Black_footed_Albatross")
        label = int(class_folder.split('.')[0])
        for img_file in os.listdir(class_path):
            if img_file.endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(class_path, img_file)
                data.append([img_path, label])

df = pd.DataFrame(data, columns=['image_path', 'label'])
df.to_csv('train_labels.csv', index=False)

print("✅ Created train_labels.csv with paths and numeric labels.")


✅ Created train_labels.csv with paths and numeric labels.


In [2]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


In [11]:
image_size = 224  # You can adjust this based on your model

# Augmentations for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Simpler transform for validation/test
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [7]:
class BirdDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['image_path']
        label = int(self.data.iloc[idx]['label'])  # Shift to 0–199 for training
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [8]:
from sklearn.model_selection import train_test_split

df = pd.read_csv('train_labels.csv')
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)


In [10]:
train_dataset = BirdDataset(csv_file='train_split.csv', transform=train_transform)
val_dataset = BirdDataset(csv_file='val_split.csv', transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4)


NameError: name 'train_transform' is not defined

In [68]:
import timm
import torch.nn as nn

model = timm.create_model('resnet50', pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 200)


In [69]:
import torch
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer (AdamW is often better for generalization)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Learning rate scheduler (cosine annealing is great with warm restarts)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 11.55 GiB of which 2.75 MiB is free. Process 1309059 has 1.04 GiB memory in use. Including non-PyTorch memory, this process has 10.47 GiB memory in use. Of the allocated memory 10.26 GiB is allocated by PyTorch, and 94.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import numpy as np

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20, patience=3):
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = (labels - 1).to(device)  # Shift 1–200 → 0–199

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        scheduler.step()

        # Validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = (labels - 1).to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = correct / total
        print(f"Train Loss: {running_loss / len(train_loader.dataset):.4f} | Val Acc: {val_acc:.4f}")

        # Early Stopping
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
            print("✅ Saved new best model")
        else:
            patience_counter += 1
            print(f"⚠️ Early stopping patience: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("⏹️ Early stopping triggered.")
                break


In [ ]:
train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=25,
    patience=4
)


In [3]:
class TestBirdDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.image_paths = sorted([os.path.join(test_dir, img)
                                   for img in os.listdir(test_dir)
                                   if img.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)['image']

        filename = os.path.basename(img_path)
        return image, filename


In [13]:
import os
test_dataset = TestBirdDataset(test_dir='data/test', transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)


In [14]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()


NameError: name 'model' is not defined